In [0]:
# 01 - Importar bibliotecas e carregar a base de features

from pyspark.sql import functions as F

from pyspark.ml import Pipeline
from pyspark.ml.feature import (
    StringIndexer,
    OneHotEncoder,
    VectorAssembler
)
from pyspark.ml.regression import (
    LinearRegression,
    RandomForestRegressor,
    GBTRegressor
)
from pyspark.ml.evaluation import RegressionEvaluator

df = spark.table(
    "workspace.projeto_integrador.gold_features_ml"
)

df_treino = df.filter(F.col("conjunto") == "treino")
df_validacao = df.filter(F.col("conjunto") == "validacao")
df_teste = df.filter(F.col("conjunto") == "teste")

print("Treino:", df_treino.count())
print("Validação:", df_validacao.count())
print("Teste reservado:", df_teste.count())

In [0]:
# 02 - Criar função de avaliação dos modelos

def calcular_metricas(df_predicoes, prediction_col):
    
    avaliador_mae = RegressionEvaluator(
        labelCol="target",
        predictionCol=prediction_col,
        metricName="mae"
    )

    avaliador_rmse = RegressionEvaluator(
        labelCol="target",
        predictionCol=prediction_col,
        metricName="rmse"
    )

    avaliador_r2 = RegressionEvaluator(
        labelCol="target",
        predictionCol=prediction_col,
        metricName="r2"
    )

    mae = avaliador_mae.evaluate(df_predicoes)
    rmse = avaliador_rmse.evaluate(df_predicoes)
    r2 = avaliador_r2.evaluate(df_predicoes)

    return mae, rmse, r2

In [0]:
# 03 - Criar modelo baseline usando o valor do ano anterior

predicoes_baseline = (
    df_validacao
    .withColumn(
        "prediction_baseline",
        F.col("valor_ano_anterior")
    )
)

mae_baseline, rmse_baseline, r2_baseline = calcular_metricas(
    predicoes_baseline,
    "prediction_baseline"
)

print("BASELINE")
print("MAE:", round(mae_baseline, 2))
print("RMSE:", round(rmse_baseline, 2))
print("R²:", round(r2_baseline, 4))

In [0]:
# 04 - Criar transformação das variáveis categóricas

indexador_uf = StringIndexer(
    inputCol="uf",
    outputCol="uf_index",
    handleInvalid="keep"
)

indexador_regiao = StringIndexer(
    inputCol="regiao_brasil",
    outputCol="regiao_index",
    handleInvalid="keep"
)

encoder = OneHotEncoder(
    inputCols=[
        "uf_index",
        "regiao_index"
    ],
    outputCols=[
        "uf_ohe",
        "regiao_ohe"
    ]
)

In [0]:
# 05 - Treinar Regressão Linear sobre o target logarítmico

features_linear = [
    "log_valor_ano_anterior",
    "log_valor_2_anos_antes",
    "log_valor_3_anos_antes",
    "log_media_3_anos",
    "variacao_percentual_historica",
    "razao_ultimo_media_3_anos",
    "uf_ohe",
    "regiao_ohe"
]

assembler_linear = VectorAssembler(
    inputCols=features_linear,
    outputCol="features_linear"
)

regressao_linear = LinearRegression(
    featuresCol="features_linear",
    labelCol="target_log",
    predictionCol="prediction_log",
    maxIter=100,
    regParam=0.1,
    elasticNetParam=0.0
)

pipeline_linear = Pipeline(
    stages=[
        indexador_uf,
        indexador_regiao,
        encoder,
        assembler_linear,
        regressao_linear
    ]
)

modelo_linear = pipeline_linear.fit(df_treino)

predicoes_linear = (
    modelo_linear
    .transform(df_validacao)

    # Retornar da escala logarítmica para a escala original
    .withColumn(
        "prediction_linear",
        F.greatest(
            F.exp(F.col("prediction_log")) - F.lit(1.0),
            F.lit(0.0)
        )
    )
)

mae_linear, rmse_linear, r2_linear = calcular_metricas(
    predicoes_linear,
    "prediction_linear"
)

print("REGRESSÃO LINEAR")
print("MAE:", round(mae_linear, 2))
print("RMSE:", round(rmse_linear, 2))
print("R²:", round(r2_linear, 4))

In [0]:
# 06 - Treinar Random Forest

features_arvore = [
    "ano",
    "valor_ano_anterior",
    "valor_2_anos_antes",
    "valor_3_anos_antes",
    "media_3_anos_anteriores",
    "diferenca_ultimo_ano",
    "variacao_percentual_historica",
    "razao_ultimo_media_3_anos",
    "uf_ohe",
    "regiao_ohe"
]

assembler_rf = VectorAssembler(
    inputCols=features_arvore,
    outputCol="features_rf"
)

random_forest = RandomForestRegressor(
    featuresCol="features_rf",
    labelCol="target",
    predictionCol="prediction_rf",
    numTrees=100,
    maxDepth=8,
    seed=42
)

pipeline_rf = Pipeline(
    stages=[
        indexador_uf,
        indexador_regiao,
        encoder,
        assembler_rf,
        random_forest
    ]
)

modelo_rf = pipeline_rf.fit(df_treino)

predicoes_rf = modelo_rf.transform(df_validacao)

mae_rf, rmse_rf, r2_rf = calcular_metricas(
    predicoes_rf,
    "prediction_rf"
)

print("RANDOM FOREST")
print("MAE:", round(mae_rf, 2))
print("RMSE:", round(rmse_rf, 2))
print("R²:", round(r2_rf, 4))

In [0]:
# 07 - Treinar Gradient Boosted Trees

assembler_gbt = VectorAssembler(
    inputCols=features_arvore,
    outputCol="features_gbt"
)

gbt = GBTRegressor(
    featuresCol="features_gbt",
    labelCol="target",
    predictionCol="prediction_gbt",
    maxIter=50,
    maxDepth=5,
    stepSize=0.05,
    seed=42
)

pipeline_gbt = Pipeline(
    stages=[
        indexador_uf,
        indexador_regiao,
        encoder,
        assembler_gbt,
        gbt
    ]
)

modelo_gbt = pipeline_gbt.fit(df_treino)

predicoes_gbt = modelo_gbt.transform(df_validacao)

mae_gbt, rmse_gbt, r2_gbt = calcular_metricas(
    predicoes_gbt,
    "prediction_gbt"
)

print("GRADIENT BOOSTED TREES")
print("MAE:", round(mae_gbt, 2))
print("RMSE:", round(rmse_gbt, 2))
print("R²:", round(r2_gbt, 4))

In [0]:
# 08 - Comparar desempenho dos modelos na validação de 2024

resultados = [
    (
        "Baseline - Ano anterior",
        float(mae_baseline),
        float(rmse_baseline),
        float(r2_baseline)
    ),
    (
        "Regressão Linear - Log",
        float(mae_linear),
        float(rmse_linear),
        float(r2_linear)
    ),
    (
        "Random Forest",
        float(mae_rf),
        float(rmse_rf),
        float(r2_rf)
    ),
    (
        "Gradient Boosted Trees",
        float(mae_gbt),
        float(rmse_gbt),
        float(r2_gbt)
    )
]

df_resultados = spark.createDataFrame(
    resultados,
    [
        "modelo",
        "mae",
        "rmse",
        "r2"
    ]
)

df_resultados = (
    df_resultados
    .withColumn(
        "mae",
        F.round("mae", 2)
    )
    .withColumn(
        "rmse",
        F.round("rmse", 2)
    )
    .withColumn(
        "r2",
        F.round("r2", 4)
    )
    .orderBy("mae")
)

display(df_resultados)

In [0]:
# 09 - Visualizar previsões dos modelos para 2024

comparacao_previsoes = (
    predicoes_rf
    .select(
        "codigo_ibge",
        "municipio",
        "uf",
        "target",
        "valor_ano_anterior",
        "prediction_rf"
    )
    .join(
        predicoes_gbt.select(
            "codigo_ibge",
            "prediction_gbt"
        ),
        on="codigo_ibge",
        how="inner"
    )
    .join(
        predicoes_linear.select(
            "codigo_ibge",
            "prediction_linear"
        ),
        on="codigo_ibge",
        how="inner"
    )
    .withColumn(
        "erro_rf",
        F.abs(
            F.col("target") -
            F.col("prediction_rf")
        )
    )
    .withColumn(
        "erro_gbt",
        F.abs(
            F.col("target") -
            F.col("prediction_gbt")
        )
    )
    .withColumn(
        "erro_linear",
        F.abs(
            F.col("target") -
            F.col("prediction_linear")
        )
    )
)

display(
    comparacao_previsoes
    .orderBy(F.desc("target"))
    .limit(30)
)

In [0]:
# 10 - Treinar Regressão Linear na escala original

features_linear_original = [
    "valor_ano_anterior",
    "valor_2_anos_antes",
    "valor_3_anos_antes",
    "media_3_anos_anteriores",
    "diferenca_ultimo_ano",
    "variacao_percentual_historica",
    "razao_ultimo_media_3_anos",
    "uf_ohe",
    "regiao_ohe"
]

assembler_linear_original = VectorAssembler(
    inputCols=features_linear_original,
    outputCol="features_linear_original"
)

regressao_linear_original = LinearRegression(
    featuresCol="features_linear_original",
    labelCol="target",
    predictionCol="prediction_linear_original",
    maxIter=100,
    regParam=0.1,
    elasticNetParam=0.0
)

pipeline_linear_original = Pipeline(
    stages=[
        indexador_uf,
        indexador_regiao,
        encoder,
        assembler_linear_original,
        regressao_linear_original
    ]
)

modelo_linear_original = pipeline_linear_original.fit(
    df_treino
)

predicoes_linear_original = (
    modelo_linear_original
    .transform(df_validacao)
    .withColumn(
        "prediction_linear_original",
        F.greatest(
            F.col("prediction_linear_original"),
            F.lit(0.0)
        )
    )
)

mae_linear_original, rmse_linear_original, r2_linear_original = (
    calcular_metricas(
        predicoes_linear_original,
        "prediction_linear_original"
    )
)

print("REGRESSÃO LINEAR - ESCALA ORIGINAL")
print("MAE:", round(mae_linear_original, 2))
print("RMSE:", round(rmse_linear_original, 2))
print("R²:", round(r2_linear_original, 4))

In [0]:
# 11 - Treinar Gradient Boosted Trees sobre o target logarítmico

features_gbt_log = [
    "log_valor_ano_anterior",
    "log_valor_2_anos_antes",
    "log_valor_3_anos_antes",
    "log_media_3_anos",
    "variacao_percentual_historica",
    "razao_ultimo_media_3_anos",
    "uf_ohe",
    "regiao_ohe"
]

assembler_gbt_log = VectorAssembler(
    inputCols=features_gbt_log,
    outputCol="features_gbt_log"
)

gbt_log = GBTRegressor(
    featuresCol="features_gbt_log",
    labelCol="target_log",
    predictionCol="prediction_gbt_log",
    maxIter=50,
    maxDepth=5,
    stepSize=0.05,
    seed=42
)

pipeline_gbt_log = Pipeline(
    stages=[
        indexador_uf,
        indexador_regiao,
        encoder,
        assembler_gbt_log,
        gbt_log
    ]
)

modelo_gbt_log = pipeline_gbt_log.fit(
    df_treino
)

predicoes_gbt_log = (
    modelo_gbt_log
    .transform(df_validacao)
    .withColumn(
        "prediction_gbt_log_original",
        F.greatest(
            F.exp(F.col("prediction_gbt_log")) - F.lit(1.0),
            F.lit(0.0)
        )
    )
)

mae_gbt_log, rmse_gbt_log, r2_gbt_log = calcular_metricas(
    predicoes_gbt_log,
    "prediction_gbt_log_original"
)

print("GRADIENT BOOSTED TREES - LOG")
print("MAE:", round(mae_gbt_log, 2))
print("RMSE:", round(rmse_gbt_log, 2))
print("R²:", round(r2_gbt_log, 4))


In [0]:
# 12 - Comparar todos os modelos na validação de 2024

resultados_completos = [
    (
        "Baseline - Ano anterior",
        float(mae_baseline),
        float(rmse_baseline),
        float(r2_baseline)
    ),
    (
        "Regressão Linear - Original",
        float(mae_linear_original),
        float(rmse_linear_original),
        float(r2_linear_original)
    ),
    (
        "Regressão Linear - Log",
        float(mae_linear),
        float(rmse_linear),
        float(r2_linear)
    ),
    (
        "Random Forest",
        float(mae_rf),
        float(rmse_rf),
        float(r2_rf)
    ),
    (
        "Gradient Boosted Trees",
        float(mae_gbt),
        float(rmse_gbt),
        float(r2_gbt)
    ),
    (
        "Gradient Boosted Trees - Log",
        float(mae_gbt_log),
        float(rmse_gbt_log),
        float(r2_gbt_log)
    )
]

df_resultados_completos = spark.createDataFrame(
    resultados_completos,
    [
        "modelo",
        "mae",
        "rmse",
        "r2"
    ]
)

df_resultados_completos = (
    df_resultados_completos
    .withColumn(
        "mae",
        F.round("mae", 2)
    )
    .withColumn(
        "rmse",
        F.round("rmse", 2)
    )
    .withColumn(
        "r2",
        F.round("r2", 4)
    )
    .orderBy("mae")
)

display(df_resultados_completos)

In [0]:
# 13 - Analisar previsões da Regressão Linear na escala original

analise_linear_original = (
    predicoes_linear_original
    .select(
        "codigo_ibge",
        "municipio",
        "uf",
        "target",
        "valor_ano_anterior",
        "prediction_linear_original"
    )
    .withColumn(
        "erro_absoluto",
        F.abs(
            F.col("target") -
            F.col("prediction_linear_original")
        )
    )
    .withColumn(
        "erro_percentual",
        F.when(
            F.col("target") > 0,
            F.round(
                (
                    F.col("erro_absoluto") /
                    F.col("target")
                ) * 100,
                2
            )
        )
    )
)

display(
    analise_linear_original
    .orderBy(F.desc("target"))
    .limit(30)
)